# 02 — Model Definition (YOLOv3 from scratch)

Darknet-53 backbone + 3-scale detection head, implemented directly in PyTorch
(`src/model.py`) rather than via a library, so every `Conv2d` is a plain
`nn.Conv2d` the Tucker-2 pipeline can enumerate and replace later — same
convention as the VGG/ResNet/ViT pipeline.

In [1]:
import sys, os, json
sys.path.append(os.path.abspath("../src"))
import torch
from model import YOLOv3, count_conv_params

with open("../checkpoints/run_config.json") as f:
    cfg = json.load(f)
NUM_CLASSES = len(cfg["class_names"])
IMG_SIZE = cfg["img_size"]
print("classes:", cfg["class_names"])

classes: ['person', 'car', 'dog', 'chair', 'bottle']


In [2]:
model = YOLOv3(num_classes=NUM_CLASSES)
total, conv = count_conv_params(model)
print(f"total params: {total:,}")
print(f"conv params:  {conv:,}  ({conv/total:.1%} of total)")
print(f"num Conv2d layers: {sum(1 for m in model.modules() if isinstance(m, torch.nn.Conv2d))}")

total params: 61,545,274
conv params:  61,492,666  (99.9% of total)
num Conv2d layers: 75


## Forward-pass shape check (3 output scales: stride 32 / 16 / 8)

In [3]:
x = torch.randn(1, 3, IMG_SIZE, IMG_SIZE)
with torch.no_grad():
    outs = model(x)
for scale_name, o in zip(["large (stride32)", "medium (stride16)", "small (stride8)"], outs):
    print(f"{scale_name}: {tuple(o.shape)}   -> anchors*(5+{NUM_CLASSES}) = {3*(5+NUM_CLASSES)} channels")

large (stride32): (1, 30, 13, 13)   -> anchors*(5+5) = 30 channels
medium (stride16): (1, 30, 26, 26)   -> anchors*(5+5) = 30 channels
small (stride8): (1, 30, 52, 52)   -> anchors*(5+5) = 30 channels


## Save an untrained checkpoint (useful as a reset point for the demo)

In [4]:
os.makedirs("../checkpoints", exist_ok=True)
torch.save(model.state_dict(), "../checkpoints/yolov3_untrained.pt")
print("saved ../checkpoints/yolov3_untrained.pt")

saved ../checkpoints/yolov3_untrained.pt


In [5]:
pip install torchinfo

Note: you may need to restart the kernel to use updated packages.


In [9]:
from torchinfo import summary

summary(model, input_size=(1, 3, IMG_SIZE, IMG_SIZE), depth=5,
        col_names=["input_size", "output_size", "num_params", "trainable"])

Layer (type:depth-idx)                        Input Shape               Output Shape              Param #                   Trainable
YOLOv3                                        [1, 3, 416, 416]          [1, 30, 13, 13]           --                        True
├─Darknet53: 1-1                              [1, 3, 416, 416]          [1, 256, 52, 52]          --                        True
│    └─Sequential: 2-1                        [1, 3, 416, 416]          [1, 32, 416, 416]         --                        True
│    │    └─Conv2d: 3-1                       [1, 3, 416, 416]          [1, 32, 416, 416]         864                       True
│    │    └─BatchNorm2d: 3-2                  [1, 32, 416, 416]         [1, 32, 416, 416]         64                        True
│    │    └─LeakyReLU: 3-3                    [1, 32, 416, 416]         [1, 32, 416, 416]         --                        --
│    └─Sequential: 2-2                        [1, 32, 416, 416]         [1, 64, 208, 208]     

In [10]:
for name, m in model.named_modules():
    if isinstance(m, torch.nn.Conv2d):
        print(f"{name:45s} k={m.kernel_size} groups={m.groups} {m.in_channels}->{m.out_channels}")

backbone.stem.0                               k=(3, 3) groups=1 3->32
backbone.stage1.0.0                           k=(3, 3) groups=1 32->64
backbone.stage1.1.conv1.0                     k=(1, 1) groups=1 64->32
backbone.stage1.1.conv2.0                     k=(3, 3) groups=1 32->64
backbone.stage2.0.0                           k=(3, 3) groups=1 64->128
backbone.stage2.1.conv1.0                     k=(1, 1) groups=1 128->64
backbone.stage2.1.conv2.0                     k=(3, 3) groups=1 64->128
backbone.stage2.2.conv1.0                     k=(1, 1) groups=1 128->64
backbone.stage2.2.conv2.0                     k=(3, 3) groups=1 64->128
backbone.stage3.0.0                           k=(3, 3) groups=1 128->256
backbone.stage3.1.conv1.0                     k=(1, 1) groups=1 256->128
backbone.stage3.1.conv2.0                     k=(3, 3) groups=1 128->256
backbone.stage3.2.conv1.0                     k=(1, 1) groups=1 256->128
backbone.stage3.2.conv2.0                     k=(3, 3) groups=1 1